From Fakeddit paper: "In addition, we used the BERT model. BERT achieves state-of-the-art results on many classification tasks, including Q&A and named entity recognition. To obtain fixed-length BERT embedding vectors, we used the bert-as-service(Xiao, 2018) tool, to map variable-length text/sentences into a 768 element array for each Reddit submission title. For our experiments, we utilized the pretrained BERT-Large, Uncased model."

Paper achieves around 82% accurcy on 3-way labels, so I should aim for the same

Use original BERT paper (Devlin et al., 2019) for fine-tuning procedure. Can be found in appendix A3:

Batch size: 16 or 32
Num epochs: 2, 3, or 4
Select best learning rate from 2, 3, 4 or 5 e-5 on val set
Dropout: 0.1
Large datasets (>100K labelled examples) far less sensitive to hyperparameter choice than small.
Fine-tuning is fast - best to run an exhaustive search across the above hyperparameters on the train/val set and see which is best.
Bert github code provides more set up details (google-research/bert):

AdamW optimiser (β1=0.9, β2=0.999, ε=1e-6)
Weight decay = 0.01
Warmup ratio (10% of total steps in published code)
(Mouratidis et al., 2025) - good paper for supporting claim that fine-tined BERT is current best practice for fake new detection. Not so good for fine-tuning methodology but they do have some interesting stuff to take into account:

They use Matthews Correlation Coefficient (MCC) and ROC_AUC alongside classic accuracy and macro F1 - good for datasets with a class imbalance (like mine).
Find that non-stemmed text performs better than stemmed and that unigrams are sufficient
This method is a bit outdated. Instead, going to fine tune bert-base-uncased end-to-end via Hugging Face Transformers library. Current best practice for BERT-based text classification.

Why bert-base-uncased?

Save GPU - BERT Large requires more GPU< likely for minimal gain. Use BERT-base instead
Uncased - using Reddit titles with inconsistent casing. Since Fakeddit's clean_title col is already lowercased, uncased makes the most sense.
Example explanation: "We use bert-base-uncased (Devlin et al., 2019: L=12, H=768, A=12, 110M parameters) rather than BERT-Large, balancing classification performance against training time within the project's compute budget. Devlin et al. (2019, §5.2) report that BERT-Large outperforms Base on most tasks but at substantially higher computational cost; for our classification setting on short titles, base-size models are standard in recent comparable work (Mouratidis et al., 2025; numerous HuggingFace baselines)."

Need to add justification from literature for using GradScaler

### 1. Setup

In [43]:
import os
import numpy as np
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import Dataset
import itertools
from sklearn.metrics import matthews_corrcoef, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

In [35]:
data_dir = os.path.join('..', 'data', 'processed', 'US')

train_path = os.path.join(data_dir, 'multimodal_train.tsv')
val_path = os.path.join(data_dir, 'multimodal_validate.tsv')

train_df = pd.read_csv(train_path, sep='\t')
val_df = pd.read_csv(val_path, sep='\t')


In [36]:
MODEL_NAME = 'bert-base-uncased'
NUM_LABELS = 3
MAX_LEN = 32 # from EDA, at least 75% of titles are under 10 words
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')
print(f'Using device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Using device: cuda


In [37]:
def tokenise(batch):
    return tokenizer(batch['clean_title'], truncation=True, padding='max_length', max_length=MAX_LEN)

def prepare_dataset(df):
    """Converts df into dataset ready for torch"""
    ds = Dataset.from_pandas(df)
    ds = ds.map(tokenise, batched=True)
    ds = ds.rename_column('3_way_label', 'labels')
    keep_cols = {'input_ids', 'attention_mask', 'labels'}
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep_cols])
    ds.set_format('torch')
    return ds

In [38]:
# small subset dataset to enable faster grid training
SUBSAMPLE_SIZE = 100_000
train_df_small, _ = train_test_split(
    train_df,
    train_size=SUBSAMPLE_SIZE,
    stratify=train_df["3_way_label"],
    random_state=42,
)

In [39]:
train_small_ds = prepare_dataset(train_df_small)
train_full_ds = prepare_dataset(train_df)
val_ds = prepare_dataset(val_df)

print(f"Train (subsample): {len(train_small_ds):,}")
print(f"Train (full): {len(train_full_ds):,}")
print(f"Val:   {len(val_ds):,}")


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/564000 [00:00<?, ? examples/s]

Map:   0%|          | 0/59342 [00:00<?, ? examples/s]

Train (subsample): 100,000
Train (full): 564,000
Val:   59,342


### 2. Training and evaluation functions

Using torch (like in Neural Networks lectures)
- Each item has 3 tensors: input_ids, attention_mask, and lables from toeknise and prepare_dataset functions 
- Batch is formed of dicts, so unpack dict and pass items to model. 

In [ ]:
def train_one_epoch(model, loader, optimiser, scheduler, scaler=None):
    """Forward pass over one epoch. Returns avg training loss"""
    model.train()
    total_loss = 0.0

    for batch in loader: 
        batch = {k: v.to(device) for k, v in batch.items()}    # move batch to device
        optimiser.zero_grad()   # clear gradients from prev step

        if scaler is not None:
            pass
        else:
            loss = model(**batch).loss  # ** unpacks the dict as keyword arguments
            loss.backward() # Compute gradients of loss
            optimiser.step()

    scheduler.step()    # Advance lr schedule

    total_loss += loss.item()

    return total_loss / len(loader) # return avg loss for epoch

include MCC as an evaluation metric because it is good for when classes are imbalanced. MCC takes into account all four quadrants of confusion matrix, across all classes. Only gives a high score if the model performs well across the whole matrix. 

In [ ]:
# evaluation metrics
def evaluate(model, loader):
    """"Evaluate model. Returns accuraxy, macro F1 and, MCC"""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels')    # Remove labels before assessing batch
            batch = {k: v.to(device) for k, v in batch.items()} 
            preds = model(**batch).logits.argmax(dim=-1).cpu().numpy()  # move back to cpu so can convert to numpy
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'macro_f1': f1_score(all_labels, all_preds, average='macro'),
        'MCC': matthews_corrcoef(all_labels, all_preds)
    }

Using 0.1 warmup ratio and 0.01 weight decay as recommended in BERT literature

In [ ]:
# full training loop

def train_model(train_ds, val_ds, batch_size, learning_rate, num_epochs, weight_decay=0.01, warmup_ratio=0.1, use_fp16=True, save_path=None):
    """Trains BERT for 'num_epochs'. Stores model with best F1"""

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    eval_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)

    optimiser = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # GradScaler enables mixed-precision training, which halves memory and speeds up GPU runs
    scaler = torch.cuda.amp.GradScaler() if (use_fp16 and device.type == 'cuda') else None

    best_f1 = -1.0
    best_epoch = -1
    best_state = None   # CPU copy of best model weights
    epoch_history = []

    progress_bar = tqdm(range(total_steps))
    for epoch in range(1, num_epochs + 1):

        train_loss = train_one_epoch(model, train_loader, optimiser, scheduler, scaler)
        val_metrics = evaluate(model, train_loader)

        print(f'Epoch {epoch} / {num_epochs}'
              f'Train loss = {train_loss:.3f}'
              f'val F1 = {val_metrics['macro_f1']:.3f}'
              f'val acc = {val_metrics['accuracy']:.3f}'
              )
        
        epoch_history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            **val_metrics
        })

        if val_metrics['macro_f1'] > best_f1:           # Save best epoch so far
            best_f1 = val_metrics['macro_f1']
            best_epch = epoch
            best_state = {k: v.cpu().cline() for k, v in model.state_dict().items()}
        
        model.load_state_dict(best_state)               # Restore best model state before continuing 

        if save_path:
            model.save_pretrained(save_path)
            tokeniser.save_pretrained(save_path)
        
        progress_bar.update()

        return {
            "best_epoch":    best_epoch,
            "best_macro_f1": best_f1,
            "epoch_history": epoch_history,
        }
        

### 3.1 Grid search on 100K sample
As recommended in BERT appendix A3, can do an exhaustive grid search over their batch size and lr hyperparamter recommendations

use train_df_small for the grid search and train_df (the full set) for the final training run

Do on the smaller training subset (to save on GPU). Then can run the full training loop on the pre-tuned configuration.

In [ ]:
BATCH_SIZES = [16, 32]
LEARNING_RATES = [5e-5, 3e-5, 2e-5]
MAX_EPOCHS = 4

grid_results = []

for batch_size, lr in itertools.product(BATCH_SIZES, LEARNING_RATES):
    loop_name = f'bs_{batch_size}_lr_{lr}'
    print(f'training run: {loop_name}')

    result = train_model(
        train_small_ds,
        val_ds,
        batch_size, 
        lr,
        num_epochs=MAX_EPOCHS
    )

    grid_results.append({
        'run_name': loop_name,
        'batch_size': batch_size, 
        'learning_rate': lr,
        **result
    })

    # save results after each loop
    with open('../results/bert_grid_search.json', 'w') as f:
        json.dump(grid_results, f, indent=2, default=str)

# sort runs by validation f1 and print summary table

grid_df = pd.DataFrame(grid_results).sort_values('best_macro_f1', ascending=False)
print('GRID SEARCH RESULTS')
print(grid_df[["run_name", "best_epoch", "best_macro_f1"]].to_string(index=False))

best = grid_df.iloc[0]
print(f"\nBest hyperparameters: batch={best['batch_size']}, "
      f"lr={best['learning_rate']:.0e}, "
      f"epochs={best['best_epoch']}")      

### 3.2 Training on full training set

Re-train from scratch using the optimal hyperparameters found by grid search

In [ ]:
final_result = train_model(
    train_full_ds,
    val_ds,
    batch_size=int(best['batch_size']),
    learning_rate=float(best["learning_rate"]),
    num_epochs=int(best["best_epoch"]),
    save_path="../models/bert_final/best"
)

print(f"\nFinal val macro-F1: {final_result['best_macro_f1']:.3f}")
print(f"Best epoch: {final_result['best_epoch']}")

### 4 Test set evaluation

In [ ]:
# test_ds = prepare_dataset(test_ds)
# test_loader = DataLoader(test_ds, batch_size=64)
# model = AutoModelForSequenceClassification.from_pretrained('../models/bert_final/best').to(device)
# test_metrics = evaluate(model, test_loader)
# print(f'Test metrics: {test_metrics}')